# Apple (AAPL) Stock Price **`Prediction`** Using LSTM

### Environment Setup

Importing the necessary libraries for data manipulation, visualization, and deep learning. I also configure the device to use Apple Silicon (MPS) for hardware acceleration.

In [1]:
# Libaray Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

import torch
import torch.nn as nn
import torch.optim as optim


from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

### Hardware Acceleration Setup (Apple Silicon)

This project is running on a **MacBook with an M2 chip**. To ensure the model trains efficiently, we utilize the **Metal Performance Shaders (MPS)** backend. 

*   **`mps`**: This allows PyTorch to leverage the GPU power of Apple Silicon.
*   **`cpu`**: Used as a fallback if the MPS backend is unavailable.

By setting the `device`, we can move our model and tensors from the system RAM to the M2's **Unified Memory** for faster computation.

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

### Data Acquisition

Using the **yfinance API**, I download historical stock data for Apple Inc. (AAPL). This provides us with daily Open, High, Low, Close, and Volume data.

In [3]:
ticker = 'AAPL'
df = yf.download(ticker, '2020-01-01')


[*********************100%***********************]  1 of 1 completed


Before modeling, I inspect the structure of our dataframe to ensure the data was downloaded correctly and to check for any missing values.

In [12]:
df.dtypes



Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

### Column Formatting

We flatten the MultiIndex columns resulting from the yfinance download to simplify data access for the preprocessing stage.

In [38]:
#Flatten the MultiIndex columns
df.columns = df.columns.get_level_values(0)

df.columns



Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')

### Visualizing the Raw Data

I use **Plotly** to create an interactive line chart of the **'Close'** price. This helps me understand the general trend and volatility of the stock over the selected time period.

In [37]:
fig = px.line(df, y='Close', title='Stock Closing Price')
fig.show()

### Data Normalization

LSTMs are sensitive to the scale of input data. We use StandardScaler to normalize the 'Close' price, ensuring the features have a mean of 0 and a standard deviation of 1.

In [14]:
scaler = StandardScaler()
df['Close'] = scaler.fit_transform(df[['Close']])

### Verification of Scaling

A quick check of the transformed 'Close' series to confirm the normalization was applied successfully.

In [15]:
df.Close

Date
2020-01-02   -1.837166
2020-01-03   -1.850399
2020-01-06   -1.839658
2020-01-07   -1.846048
2020-01-08   -1.824296
                ...   
2026-05-01    2.068371
2026-05-04    2.006142
2026-05-05    2.144323
2026-05-06    2.206928
2026-05-07    2.205612
Name: Close, Length: 1595, dtype: float64

### Data Windowing and Tensor Preparation

To train the model, we create a Sliding Window of 30 days.

Sequencing: We transform the 1D price series into a 3D shape (Samples, Time_Steps, Features).

Splitting: 80% of the data is reserved for training, and 20% for testing.

Tensors: Data is converted into PyTorch tensors and moved to the GPU/MPS device.

In [22]:
# I want to consider 30 days
seq_length = 30
# Preparing an empty liist for the data
data = []
# I will take the examples, I will take the first days and then try to predict the last days
for i in range(len(df) -seq_length):
    data.append(df.Close[i:i+seq_length])

# Converting the data into a numpy array of data, just so its easier to handle

data = np.array(data)
data = data.reshape(data.shape[0], data.shape[1], 1)

#Defining the training size
train_size = int(0.8 * len(data))

# X = all days except the last one in the sequence
# Y = only the last day (the target)
x_train = torch.from_numpy(data[:train_size, :-1, :]).type(torch.Tensor).to(device)
y_train = torch.from_numpy(data[:train_size, -1, :]).type(torch.Tensor).to(device)

x_test = torch.from_numpy(data[train_size:, :-1, :]).type(torch.Tensor).to(device)
y_test = torch.from_numpy(data[train_size:, -1, :]).type(torch.Tensor).to(device)

### Defining the Prediction Model (LSTM)

I construct a Many-to-One LSTM architecture designed for time-series forecasting.

How the Model is Built:

LSTM Layers: A stacked LSTM with 2 layers captures complex temporal dependencies.

Hidden Dimension: We use 32 hidden units to balance model capacity and prevent overfitting.

Forward Pass: The model processes a sequence of 29 days. We extract the output of the final hidden state (out[:, -1, :]) representing the model's "summary" of the month.

Fully Connected Layer: This final linear layer maps the LSTM's summary to a single continuous value—the predicted price for the next day.

In [23]:
class PredictionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(PredictionModel, self).__init__()

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True) # LSTM Archiecture - we can add , dropuout = 02
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=device)

        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))
        out = self.fc(out[:, -1, :])


        return out

### Model Instantiation

I initialize the PredictionModel with our defined dimensions and move the entire architecture to the active device (MPS).

In [24]:
model = PredictionModel(input_dim=1, hidden_dim=32, num_layers=2, output_dim=1).to(device)

Criteria For Training

### Loss Function and Optimizer

* Criterion: We use Mean Squared Error (MSE), which is standard for regression tasks.

* Optimizer: The Adam optimizer is used with a learning rate of 0.01 for efficient, adaptive weight updates.

In [25]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

### Training Loop

The model iterates through the training data for 200 epochs. In each iteration, it performs a forward pass, calculates the error, backpropagates the gradients, and updates the weights.

In [30]:
num_epochs = 200

for i in range(num_epochs):
    y_train_pred = model(x_train)

    loss = criterion(y_train_pred, y_train)

    if i % 25 == 0:
        print(i, loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

0 0.0026831943541765213
25 0.004767132457345724
50 0.002772518200799823
75 0.0025960628408938646
100 0.002578927204012871
125 0.0025716687086969614
150 0.0025671962648630142
175 0.002564015332609415


Now Visualizing how the data performs on unseen data

### Model Evaluation and Inverse Scaling

I switch the model to .eval() mode and generate predictions on the test set. To make the results interpretable, I use .inverse_transform() to convert the scaled values back into their original US Dollar amounts.

In [31]:
model.eval()
y_test_pred = model(x_test)

y_train_pred = scaler.inverse_transform(y_train_pred.detach().cpu().numpy())
y_train = scaler.inverse_transform(y_train.detach().cpu().numpy())
y_test_pred = scaler.inverse_transform(y_test_pred.detach().cpu().numpy())
y_test = scaler.inverse_transform(y_test.detach().cpu().numpy())

In [32]:
train_rmse = root_mean_squared_error(y_train[:, 0], y_train_pred[:, 0])
test_rmse = root_mean_squared_error(y_test[:, 0], y_test_pred[:, 0])

In [33]:
train_rmse

2.6921348571777344

In [34]:
test_rmse

5.207521915435791

### Final Visualization

I create a dual-panel visualization. The top plot compares the Actual vs. Predicted prices to check for trend alignment, while the bottom plot tracks the Prediction Error over time to identify specific periods of high volatility.

In [36]:
from plotly.subplots import make_subplots

# 1. Create a subplot figure: 2 rows, 1 column
# row_heights gives the top plot 75% of the space, similar to your gridspec
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=(f"{ticker} Stock Price Prediction", "Prediction Error"),
    row_heights=[0.7, 0.3]
)

# 2. Get the dates for the x-axis
dates = df.iloc[-len(y_test):].index

# 3. Add Actual Price Trace (Top Plot)
fig.add_trace(
    go.Scatter(x=dates, y=y_test.flatten(), name='Actual Price', line=dict(color='blue')),
    row=1, col=1
)

# 4. Add Predicted Price Trace (Top Plot)
fig.add_trace(
    go.Scatter(x=dates, y=y_test_pred.flatten(), name='Predicted Price', line=dict(color='green')),
    row=1, col=1
)

# 5. Add Prediction Error Trace (Bottom Plot)
error_values = abs(y_test.flatten() - y_test_pred.flatten())
fig.add_trace(
    go.Scatter(x=dates, y=error_values, name='Prediction Error', line=dict(color='red'), fill='tozeroy'),
    row=2, col=1
)

# 6. Add the RMSE Horizontal Line (Bottom Plot)
fig.add_hline(
    y=test_rmse, 
    line_dash="dash", 
    line_color="blue", 
    annotation_text=f"RMSE: {test_rmse:.2f}",
    row=2, col=1
)

# 7. Update layout details
fig.update_layout(
    height=800, 
    width=1000, 
    template="plotly_white",
    hovermode="x unified",
    showlegend=True
)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Error", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()